# Projeto: Normalização e Preparação dos Dados Meteorológicos – Weather


## Descrição Geral
Este notebook trata da normalização e preparação dos dados brutos do Weather.com, extraídos da camada bronze, para análise e consumo pelas próximas camadas (silver/gold).<br>
Antes de serem processados neste notebook, os dados passam por uma etapa automatizada de validação com Great Expectations (GX). Apenas os registros validados com sucesso são utilizados nas transformações abaixo.<br>
As etapas incluem renomeação de colunas, tratamento de tipos de dados, separação de campos compostos e padronização de formatos.

## Objetivos

- Padronizar os nomes das colunas conforme convenção definida.
- Realizar o ajuste de tipos de dados para colunas de datas, timestamps e decimal.
- Realizar a remoção de dados duplicados
- Separar colunas compostas (como precipitação e probabilidade).
- Preparar o DataFrame para persistência na camada silver.

## Tecnologias Utilizadas

- Apache Spark (PySpark)
- Databricks Notebooks
- Delta Lake


## Fonte de Dados

- **Localização:** [Blob Storage - previsoes_weather](grupo01.bronze.weather)
- **Formato:** Delta
- **Origem:** Tabela bronze criada a partir da ingestão do dataset Weather

---


### Histórico de Atualizações

| Data       | Responsável    | Versão | Descrição                                            |
|------------|----------------|--------|------------------------------------------------------|
| 16/05/2025 | Evelise Dias   | 1.0    | Pipeline de ingestão do Weather  |
| 16/05/2025 | Evelise Dias   | 1.1    | Criação da tabela individual bronze no medallion     |
| 19/05/2025 | Evelise Dias   | 1.2    | Atualização de documentação     |
| 25/05/2025 | Evelise Dias   | 1.3    | Ajuste do pipeline para ler apenas dados validados pelo GX                     |
| 26/05/2025 | Evelise Dias   | 1.4    | Atualização da documentação para refletir a origem validada dos dados          |

In [0]:
%run ../utils/utils_run_silver

In [0]:
%run ./queries_silver

In [0]:
pasta_validos = lakehouse_logs_gx + "/bronze/weather/validos/weather"
df_weather = ler_tabelas(pasta_validos, fromTable=False)

In [0]:
df_weather_renomear_colunas = renomear_colunas(df_weather, CONVENCAO_COLUNAS_WEATHER)

In [0]:
df_weather_ajustar_tipagem_datas = alterar_tipos(df_weather_renomear_colunas, ["dt_previsao"], "date")
df_weather_ajustar_tipagem_datas = alterar_tipos(df_weather_ajustar_tipagem_datas,["dt_carga"],"timestamp",data_formato="yyyy-MM-dd HH:mm:ss")

In [0]:
df_weather_limpar_celulas = tratar_celulas(df_weather_ajustar_tipagem_datas)

In [0]:
df_weather_ajustar_tipagem_inteiros = alterar_tipos(df_weather_limpar_celulas, ["temp_max", "temp_min", "umid_max", "umid_min", "precipitacao", "vel_vento", "prob_precipitacao"], "decimal")

In [0]:
df_sem_duplicatas = remover_duplicatas(df_weather_ajustar_tipagem_inteiros, ["dt_carga", "dt_previsao", "sig_uf","nom_municipio", "flg_fonte_previsao"])

In [0]:
df_com_id = gera_primary_key(df_sem_duplicatas)

In [0]:
spark.sql(create_table_weather_silver)

In [0]:
inserir_com_merge(dataframe = df_com_id, tabela_destino="grupo01.silver.weather", camposChave=["id"])